<a href="https://colab.research.google.com/github/Anmolsharma44/tndrr-ai/blob/main/Task_3_Local_Inference_optimization/A%5B3%5DQwen2_5_0_5B_Instruct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import time
import torch
import warnings
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Deprecation warnings ko ignore karne ke liye
warnings.filterwarnings("ignore", category=UserWarning)

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokener = AutoTokenizer.from_pretrained(model_id)

prompts = [
    "Extract booking: Haircut for Rahul at 10 AM tomorrow.",
    "Explain KV cache in two sentences.",
    "Summarize why quantization reduces GPU memory usage.",
    "Extract booking: Beard trim for Amit on Friday at 5 PM.",
    "What is the difference between FP16 and INT4?"
]

def run_benchmark(quantized=False):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device.upper()}")

    if device == "cuda":
        torch.cuda.empty_cache()

    if quantized:
        if device == "cpu":
            print("[4-Bit Quantized] Skip: 4-bit quantization is not supported on CPU.")
            return
        bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
        label = "4-Bit Quantized"
    else:
        # CPU supports float32 better unless bfloat16 is specified
        dtype = torch.float16 if device == "cuda" else torch.float32
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, device_map="auto" if device == "cuda" else None).to(device)
        label = f"16-Bit ({'FP16' if device == 'cuda' else 'FP32'}"

    total_tokens = 0
    start_time = time.time()

    for p in prompts:
        inputs = tokener(p, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=40, do_sample=False)
        total_tokens += (out.shape[1] - inputs["input_ids"].shape[1])

    elapsed = time.time() - start_time
    if device == "cuda":
        vram_mb = torch.cuda.memory_allocated() / (1024 ** 2)
        tps = total_tokens / elapsed
        print(f"[{label}] Memory Allocated: {vram_mb:.2f} MB | Speed: {tps:.2f} tokens/sec")
    else:
        tps = total_tokens / elapsed
        print(f"[{label}] CPU Inference Completed | Speed: {tps:.2f} tokens/sec")

if __name__ == "__main__":
    print("Running Local Inference Optimization Benchmark...")
    run_benchmark(quantized=False)
    run_benchmark(quantized=True)

Running Local Inference Optimization Benchmark...
Using device: CPU


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[16-Bit (FP32] CPU Inference Completed | Speed: 4.26 tokens/sec
Using device: CPU
[4-Bit Quantized] Skip: 4-bit quantization is not supported on CPU.
